# E-Commerce Sales & Customer Analytics
## End-to-End Data Analysis Notebook

**Objective:** Import, clean, analyze and visualize e-commerce sales data to identify revenue, profitability, product, regional and customer-segment insights.


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## 2. Load Raw Data


In [ ]:
sales = pd.read_csv('../data/sales_raw.csv')
customers = pd.read_csv('../data/customers.csv')

print('Sales shape:', sales.shape)
print('Customer shape:', customers.shape)
sales.head()


## 3. Data Quality Assessment
Check data types, missing values, duplicates and suspicious numeric values before analysis.

In [ ]:
display(sales.info())
print('\nMissing values:')
display(sales.isna().sum().sort_values(ascending=False))
print('\nDuplicate order IDs:', sales['order_id'].duplicated().sum())
print('\nInvalid unit prices:', (sales['unit_price'] <= 0).sum())


## 4. Data Cleaning
Missing discounts are filled with the median. Blank regions become `Unknown`. Invalid unit prices are replaced using the category median. Revenue and profit are recalculated after cleaning.

In [ ]:
df = sales.copy()

df['region'] = df['region'].replace(r'^\s*$', np.nan, regex=True).fillna('Unknown')
df['discount'] = df['discount'].fillna(df['discount'].median())
df.loc[df['unit_price'] <= 0, 'unit_price'] = np.nan
df['unit_price'] = df['unit_price'].fillna(df.groupby('category')['unit_price'].transform('median'))

df['order_date'] = pd.to_datetime(df['order_date'])
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'])
df['profit'] = df['revenue'] - df['cost']
df['profit_margin'] = df['profit'] / df['revenue']
df['month'] = df['order_date'].dt.to_period('M').astype(str)

display(df.head())


## 5. KPI Analysis

In [ ]:
kpis = {
    'Revenue': df['revenue'].sum(),
    'Profit': df['profit'].sum(),
    'Orders': df['order_id'].nunique(),
    'Average Order Value': df.groupby('order_id')['revenue'].sum().mean(),
    'Profit Margin': df['profit'].sum() / df['revenue'].sum(),
    'Customers': df['customer_id'].nunique()
}
pd.Series(kpis)


## 6. Monthly Sales Trend

In [ ]:
monthly = df.groupby('month').agg(revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique')).reset_index()
plt.figure(figsize=(10,5))
plt.plot(monthly['month'], monthly['revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month'); plt.ylabel('Revenue')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


## 7. Regional and Category Performance

In [ ]:
region_summary = df.groupby('region').agg(revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique')).sort_values('revenue', ascending=False)
category_summary = df.groupby('category').agg(revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique')).sort_values('revenue', ascending=False)
display(region_summary)
display(category_summary)


## 8. Product and Customer Segment Analysis

In [ ]:
top_products = df.groupby('product').agg(revenue=('revenue','sum'), profit=('profit','sum'), quantity=('quantity','sum')).sort_values('revenue', ascending=False).head(10)
segment_summary = df.groupby('segment').agg(revenue=('revenue','sum'), profit=('profit','sum'), customers=('customer_id','nunique')).sort_values('revenue', ascending=False)
display(top_products)
display(segment_summary)


## 9. Business Insights

- Identify the strongest revenue region and category.
- Compare revenue concentration with profit contribution.
- Identify top products for sales prioritization.
- Evaluate customer segments by revenue and customer count.
- Use monthly trends to identify seasonality or growth periods.


## 10. Export Analysis-Ready Dataset


In [ ]:
df.to_csv('../data/sales_clean.csv', index=False)
print('Saved cleaned dataset to ../data/sales_clean.csv')
